# **Hypothesis Testing**

The goal of this file is to determine the effect a home run has on price in the Los Angeles Angels (LAA) and Los Angeles Dodgers (LAD) sports trading markets. Additional analysis was performed for the Chicago White Sox (CWS) and the San Francisco Giants (SF).

## Data Description

### Dependent Variables

- **price_at_hr**: The price of a trade at the time a home run was hit, where all prices represent the probability of the home team winning (used only in regression models with generic and forward data).

- **d_price_w $\alpha$ _b $\beta$**: The double binned mean price at the time of a home run.
    - $\alpha$ is the window size: $\alpha \in \{1, 2, 3, 4, 5, 6\}$
    - $\beta$ is the bin size: $\beta \in \{1, 2, 3, 4, 5, 6\}$

- **laa_homerun_dummy**: Binary indicator equal to 1 if LAA hit a home run at that timestamp, 0 otherwise.

- **lad_homerun_dummy**: Binary indicator equal to 1 if LAD hit a home run at that timestamp, 0 otherwise.

- **cws_homerun_dummy**: Binary indicator equal to 1 if CWS hit a home run at that timestamp, 0 otherwise.

- **sf_homerun_dummy**: Binary indicator equal to 1 if SF hit a home run at that timestamp, 0 otherwise.

- **runners_on_base**: The number of runners on base when the home run was hit.

- **inning**: The inning in which the home run was hit.

- **laa_score**: LAA's score when the home run was hit.

- **lad_score**: LAD's score when the home run was hit.

- **cws_score**: CWS's score when the home run was hit.

- **sf_score**: SF's score when the home run was hit.

- **opp_score**: The opponent's score when the home run was hit.

### Independent Variables

- **g_px_chg**: The generic price change measuring market reaction to a home run (used only in regression models with generic data).

- **f_px_chg_w $\alpha$ _b $\beta$**: The forward binned mean price change measuring market reaction to a home run (used only in regression models with forward binned data).

    - $\alpha$ is the window size: $\alpha \in \{1, 2, 3, 4, 5, 6\}$
    - $\beta$ is the bin size: $\beta \in \{1, 2, 3, 4, 5, 6\}$

- **d_px_chg_w $\alpha$ _b $\beta$**: The double binned mean price change measuring market reaction to a home run (used only in regression models with double binned data).

    - $\alpha$ is the window size: $\alpha \in \{1, 2, 3, 4, 5, 6\}$
    - $\beta$ is the bin size: $\beta \in \{1, 2, 3, 4, 5, 6\}$

In [62]:
import pandas as pd
from utils.hypothesis_testing import get_model_stats, graph_model_stats

In [63]:
# Load in feature engineered data frames

laa_generic_df = pd.read_parquet("../data/processed/laa_generic_data.parquet")
laa_forward_df = pd.read_parquet("../data/processed/laa_forward_data.parquet")
laa_double_df = pd.read_parquet("../data/processed/laa_double_data.parquet")

lad_generic_df = pd.read_parquet("../data/processed/lad_generic_data.parquet")
lad_forward_df = pd.read_parquet("../data/processed/lad_forward_data.parquet")
lad_double_df = pd.read_parquet("../data/processed/lad_double_data.parquet")

cws_generic_df = pd.read_parquet("../data/processed/cws_generic_data.parquet")
cws_forward_df = pd.read_parquet("../data/processed/cws_forward_data.parquet")
cws_double_df = pd.read_parquet("../data/processed/cws_double_data.parquet")

sf_generic_df = pd.read_parquet("../data/processed/sf_generic_data.parquet")
sf_forward_df = pd.read_parquet("../data/processed/sf_forward_data.parquet")
sf_double_df = pd.read_parquet("../data/processed/sf_double_data.parquet")

In [64]:
# Rename price column

laa_generic_df = laa_generic_df.rename(columns={'yes_price_dollars': 'price_at_hr'})
laa_forward_df = laa_forward_df.rename(columns={'yes_price_dollars': 'price_at_hr'})

lad_generic_df = lad_generic_df.rename(columns={'yes_price_dollars': 'price_at_hr'})
lad_forward_df = lad_forward_df.rename(columns={'yes_price_dollars': 'price_at_hr'})

cws_generic_df = cws_generic_df.rename(columns={'yes_price_dollars': 'price_at_hr'})
cws_forward_df = cws_forward_df.rename(columns={'yes_price_dollars': 'price_at_hr'})

sf_generic_df = sf_generic_df.rename(columns={'yes_price_dollars': 'price_at_hr'})
sf_forward_df = sf_forward_df.rename(columns={'yes_price_dollars': 'price_at_hr'})

## Generic Target Regression Analysis

In [65]:
# Define features and targets

laa_generic_features = [
    'laa_homerun_dummy', 'price_at_hr', 'runners_on_base', 
    'inning', 'laa_score', 'opp_score'
]
lad_generic_features = [
    'lad_homerun_dummy', 'price_at_hr', 'runners_on_base', 
    'inning', 'lad_score', 'opp_score'
]
cws_generic_features = [
    'cws_homerun_dummy', 'price_at_hr', 'runners_on_base', 
    'inning', 'cws_score', 'opp_score'
]
sf_generic_features = [
    'sf_homerun_dummy', 'price_at_hr', 'runners_on_base', 
    'inning', 'sf_score', 'opp_score'
]

generic_targets = ['g_px_chg']

# Look at a subset of the features, and drop any price change data if missing
laa_generic_df = laa_generic_df[generic_targets + laa_generic_features].dropna(subset=generic_targets)
lad_generic_df = lad_generic_df[generic_targets + lad_generic_features].dropna(subset=generic_targets)
cws_generic_df = cws_generic_df[generic_targets + cws_generic_features].dropna(subset=generic_targets)
sf_generic_df = sf_generic_df[generic_targets + sf_generic_features].dropna(subset=generic_targets)

laa_generic_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 339 entries, 0 to 362
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   g_px_chg           339 non-null    float64
 1   laa_homerun_dummy  339 non-null    int64  
 2   price_at_hr        339 non-null    float64
 3   runners_on_base    339 non-null    float64
 4   inning             339 non-null    float64
 5   laa_score          339 non-null    float64
 6   opp_score          339 non-null    float64
dtypes: float64(6), int64(1)
memory usage: 21.2 KB


In [66]:
lad_generic_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 393 entries, 0 to 402
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   g_px_chg           393 non-null    float64
 1   lad_homerun_dummy  393 non-null    int64  
 2   price_at_hr        393 non-null    float64
 3   runners_on_base    393 non-null    float64
 4   inning             393 non-null    float64
 5   lad_score          393 non-null    float64
 6   opp_score          393 non-null    float64
dtypes: float64(6), int64(1)
memory usage: 24.6 KB


In [67]:
cws_generic_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 269 entries, 0 to 286
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   g_px_chg           269 non-null    float64
 1   cws_homerun_dummy  269 non-null    int64  
 2   price_at_hr        269 non-null    float64
 3   runners_on_base    269 non-null    float64
 4   inning             269 non-null    float64
 5   cws_score          269 non-null    float64
 6   opp_score          269 non-null    float64
dtypes: float64(6), int64(1)
memory usage: 16.8 KB


In [68]:
sf_generic_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 264 entries, 0 to 271
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   g_px_chg          264 non-null    float64
 1   sf_homerun_dummy  264 non-null    int64  
 2   price_at_hr       264 non-null    float64
 3   runners_on_base   264 non-null    float64
 4   inning            264 non-null    float64
 5   sf_score          264 non-null    float64
 6   opp_score         264 non-null    float64
dtypes: float64(6), int64(1)
memory usage: 16.5 KB


### Model 1

$$\mathbb{E}[Y \mid X_1] = \beta_0 + \beta_1 X_1$$

where:

- $Y$ is **g_px_chg**, and

- $X_1$ is **laa_homerun_dummy**, **lad_homerun_dummy**, **cws_homerun_dummy** or **sf_homerun_dummy**.

In [69]:
# Regression modeling

laa_m1_stats = get_model_stats(laa_generic_df, 'generic', generic_targets, ['laa_homerun_dummy'])
lad_m1_stats = get_model_stats(lad_generic_df, 'generic', generic_targets, ['lad_homerun_dummy'])
cws_m1_stats = get_model_stats(cws_generic_df, 'generic', generic_targets, ['cws_homerun_dummy'])
sf_m1_stats = get_model_stats(sf_generic_df, 'generic', generic_targets, ['sf_homerun_dummy'])

In [70]:
laa_m1_stats[0]

{'r_squared': 0.31077717017447304,
 'beta_const': -0.07503144654088069,
 'beta_laa_homerun_dummy': 0.1243647798742141,
 'p_value_const': 1.7253936967108168e-21,
 'p_value_laa_homerun_dummy': 4.491420910082999e-29}

In [71]:
lad_m1_stats[0]

{'r_squared': 0.15419188229957015,
 'beta_const': -0.047208588957055195,
 'beta_lad_homerun_dummy': 0.07733902373966389,
 'p_value_const': 5.8179312996570616e-11,
 'p_value_lad_homerun_dummy': 6.128089428464981e-16}

In [72]:
cws_m1_stats[0]

{'r_squared': 0.3276419383021755,
 'beta_const': -0.06828671328671333,
 'beta_cws_homerun_dummy': 0.1267787767787769,
 'p_value_const': 5.099896038579859e-17,
 'p_value_cws_homerun_dummy': 8.165968203430569e-25}

In [73]:
sf_m1_stats[0]

{'r_squared': 0.2802396601239332,
 'beta_const': -0.05728000000000009,
 'beta_sf_homerun_dummy': 0.10012172661870522,
 'p_value_const': 5.1211499156065135e-14,
 'p_value_sf_homerun_dummy': 1.8034286755368975e-20}

### Model 2

$$\mathbb{E}[Y \mid X_1, X_2, X_3, X_4, X_5, X_6] = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_3 + \beta_4 X_4 + \beta_5 X_5 + \beta_6 X_6$$

where:

- $Y$ is **g_px_chg**,

- $X_1$ is either **laa_homerun_dummy**, **lad_homerun_dummy**, **cws_homerun_dummy** or **sf_homerun_dummy**,

- $X_2$ is the **price_at_hr**,

- $X_3$ is the **runners_on_base**,

- $X_4$ is the **inning**,

- $X_5$ is either **laa_score**, **lad_score**, **cws_score**, or **sf_score**, and

- $X_6$ is the **opp_score**.

In [74]:
# Regression modeling

laa_m2_stats = get_model_stats(
    laa_generic_df,
    'generic',
    generic_targets,
    laa_generic_features
)

lad_m2_stats = get_model_stats(
    lad_generic_df,
    'generic',
    generic_targets,
    lad_generic_features
)

cws_m2_stats = get_model_stats(
    cws_generic_df,
    'generic',
    generic_targets,
    cws_generic_features
)

sf_m2_stats = get_model_stats(
    sf_generic_df,
    'generic',
    generic_targets,
    sf_generic_features
)

In [75]:
laa_m2_stats[0]

{'r_squared': 0.40385183365674704,
 'beta_const': 0.01258354747071811,
 'beta_laa_homerun_dummy': 0.15319263592937604,
 'beta_price_at_hr': -0.2257791048265621,
 'beta_runners_on_base': 0.007933439502990266,
 'beta_inning': 0.00023594387465820704,
 'beta_laa_score': 0.01948700218781856,
 'beta_opp_score': -0.01924160217407792,
 'p_value_const': 0.46102092917729076,
 'p_value_laa_homerun_dummy': 1.464862297126075e-37,
 'p_value_price_at_hr': 1.9540589623357176e-11,
 'p_value_runners_on_base': 0.20735622082350985,
 'p_value_inning': 0.9261924927486309,
 'p_value_laa_score': 4.311558567687196e-06,
 'p_value_opp_score': 1.939799991526553e-06}

In [76]:
lad_m2_stats[0]

{'r_squared': 0.26080632629433476,
 'beta_const': 0.05493926214944485,
 'beta_lad_homerun_dummy': 0.10866774835420345,
 'beta_price_at_hr': -0.1964104504685484,
 'beta_runners_on_base': 0.0020413868309797677,
 'beta_inning': -0.0011156808981294649,
 'beta_lad_score': 0.013100917308099473,
 'beta_opp_score': -0.013611242719994665,
 'p_value_const': 0.0031712426144306642,
 'p_value_lad_homerun_dummy': 1.947660155399807e-25,
 'p_value_price_at_hr': 5.4385328388245644e-12,
 'p_value_runners_on_base': 0.707348820981142,
 'p_value_inning': 0.5788359333220929,
 'p_value_lad_score': 5.6106294779989595e-06,
 'p_value_opp_score': 4.5200644147249166e-05}

In [77]:
cws_m2_stats[0]

{'r_squared': 0.4449772071906145,
 'beta_const': 0.014028874523929443,
 'beta_cws_homerun_dummy': 0.16107659451239464,
 'beta_price_at_hr': -0.25768540518400385,
 'beta_runners_on_base': 0.0016336895313543336,
 'beta_inning': 0.00016276010102994447,
 'beta_cws_score': 0.023128665980242002,
 'beta_opp_score': -0.016934133901289943,
 'p_value_const': 0.4208341766980285,
 'p_value_cws_homerun_dummy': 1.1550572715873456e-34,
 'p_value_price_at_hr': 3.8899116669730116e-12,
 'p_value_runners_on_base': 0.8096161185077154,
 'p_value_inning': 0.9520597627231179,
 'p_value_cws_score': 2.0039652421238773e-08,
 'p_value_opp_score': 2.4642421963423975e-05}

In [78]:
sf_m2_stats[0]

{'r_squared': 0.37030535842011425,
 'beta_const': 0.02876028530417806,
 'beta_sf_homerun_dummy': 0.12744384513067275,
 'beta_price_at_hr': -0.18119986936667484,
 'beta_runners_on_base': 0.00748744105736025,
 'beta_inning': -0.0032616655814174155,
 'beta_sf_score': 0.015376132838474294,
 'beta_opp_score': -0.013263400844297446,
 'p_value_const': 0.1187054318375566,
 'p_value_sf_homerun_dummy': 7.059636315202923e-27,
 'p_value_price_at_hr': 2.5089885728500783e-08,
 'p_value_runners_on_base': 0.22693367854937516,
 'p_value_inning': 0.2172340126785116,
 'p_value_sf_score': 9.092172204272291e-05,
 'p_value_opp_score': 0.0008752153688058918}

## Forward Target Regression Analysis

In [79]:
# Define features and targets

laa_forward_features = [
    'laa_homerun_dummy', 'price_at_hr', 'runners_on_base', 
    'inning', 'laa_score', 'opp_score'
]
lad_forward_features = [
    'lad_homerun_dummy', 'price_at_hr', 'runners_on_base', 
    'inning', 'lad_score', 'opp_score'
]
cws_forward_features = [
    'cws_homerun_dummy', 'price_at_hr', 'runners_on_base', 
    'inning', 'cws_score', 'opp_score'
]
sf_forward_features = [
    'sf_homerun_dummy', 'price_at_hr', 'runners_on_base', 
    'inning', 'sf_score', 'opp_score'
]

forward_targets = [col for col in laa_forward_df.columns if 'f_px_chg' in col]

# Look at a subset of the features, and drop any price change data if missing
laa_forward_df = laa_forward_df[forward_targets + laa_forward_features].dropna(subset=forward_targets)
lad_forward_df = lad_forward_df[forward_targets + lad_forward_features].dropna(subset=forward_targets)
cws_forward_df = cws_forward_df[forward_targets + cws_forward_features].dropna(subset=forward_targets)
sf_forward_df = sf_forward_df[forward_targets + sf_forward_features].dropna(subset=forward_targets)

laa_forward_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 315 entries, 0 to 362
Data columns (total 42 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   f_px_chg_w1_b1     315 non-null    float64
 1   f_px_chg_w1_b2     315 non-null    float64
 2   f_px_chg_w1_b3     315 non-null    float64
 3   f_px_chg_w1_b4     315 non-null    float64
 4   f_px_chg_w1_b5     315 non-null    float64
 5   f_px_chg_w1_b6     315 non-null    float64
 6   f_px_chg_w2_b1     315 non-null    float64
 7   f_px_chg_w2_b2     315 non-null    float64
 8   f_px_chg_w2_b3     315 non-null    float64
 9   f_px_chg_w2_b4     315 non-null    float64
 10  f_px_chg_w2_b5     315 non-null    float64
 11  f_px_chg_w2_b6     315 non-null    float64
 12  f_px_chg_w3_b1     315 non-null    float64
 13  f_px_chg_w3_b2     315 non-null    float64
 14  f_px_chg_w3_b3     315 non-null    float64
 15  f_px_chg_w3_b4     315 non-null    float64
 16  f_px_chg_w3_b5     315 non-null

In [80]:
lad_forward_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 365 entries, 0 to 402
Data columns (total 42 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   f_px_chg_w1_b1     365 non-null    float64
 1   f_px_chg_w1_b2     365 non-null    float64
 2   f_px_chg_w1_b3     365 non-null    float64
 3   f_px_chg_w1_b4     365 non-null    float64
 4   f_px_chg_w1_b5     365 non-null    float64
 5   f_px_chg_w1_b6     365 non-null    float64
 6   f_px_chg_w2_b1     365 non-null    float64
 7   f_px_chg_w2_b2     365 non-null    float64
 8   f_px_chg_w2_b3     365 non-null    float64
 9   f_px_chg_w2_b4     365 non-null    float64
 10  f_px_chg_w2_b5     365 non-null    float64
 11  f_px_chg_w2_b6     365 non-null    float64
 12  f_px_chg_w3_b1     365 non-null    float64
 13  f_px_chg_w3_b2     365 non-null    float64
 14  f_px_chg_w3_b3     365 non-null    float64
 15  f_px_chg_w3_b4     365 non-null    float64
 16  f_px_chg_w3_b5     365 non-null

In [81]:
cws_forward_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 248 entries, 4 to 285
Data columns (total 42 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   f_px_chg_w1_b1     248 non-null    float64
 1   f_px_chg_w1_b2     248 non-null    float64
 2   f_px_chg_w1_b3     248 non-null    float64
 3   f_px_chg_w1_b4     248 non-null    float64
 4   f_px_chg_w1_b5     248 non-null    float64
 5   f_px_chg_w1_b6     248 non-null    float64
 6   f_px_chg_w2_b1     248 non-null    float64
 7   f_px_chg_w2_b2     248 non-null    float64
 8   f_px_chg_w2_b3     248 non-null    float64
 9   f_px_chg_w2_b4     248 non-null    float64
 10  f_px_chg_w2_b5     248 non-null    float64
 11  f_px_chg_w2_b6     248 non-null    float64
 12  f_px_chg_w3_b1     248 non-null    float64
 13  f_px_chg_w3_b2     248 non-null    float64
 14  f_px_chg_w3_b3     248 non-null    float64
 15  f_px_chg_w3_b4     248 non-null    float64
 16  f_px_chg_w3_b5     248 non-null

In [82]:
sf_forward_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 240 entries, 0 to 271
Data columns (total 42 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   f_px_chg_w1_b1    240 non-null    float64
 1   f_px_chg_w1_b2    240 non-null    float64
 2   f_px_chg_w1_b3    240 non-null    float64
 3   f_px_chg_w1_b4    240 non-null    float64
 4   f_px_chg_w1_b5    240 non-null    float64
 5   f_px_chg_w1_b6    240 non-null    float64
 6   f_px_chg_w2_b1    240 non-null    float64
 7   f_px_chg_w2_b2    240 non-null    float64
 8   f_px_chg_w2_b3    240 non-null    float64
 9   f_px_chg_w2_b4    240 non-null    float64
 10  f_px_chg_w2_b5    240 non-null    float64
 11  f_px_chg_w2_b6    240 non-null    float64
 12  f_px_chg_w3_b1    240 non-null    float64
 13  f_px_chg_w3_b2    240 non-null    float64
 14  f_px_chg_w3_b3    240 non-null    float64
 15  f_px_chg_w3_b4    240 non-null    float64
 16  f_px_chg_w3_b5    240 non-null    float64
 17  f_

### Model 1

$$\mathbb{E}[Y \mid X_1] = \beta_0 + \beta_1 X_1$$

where:

- $Y$ is **f_px_chg_w $\alpha$ _b $\beta$**, and

- $X_1$ is either **laa_homerun_dummy**, **lad_homerun_dummy**, **cws_homerun_dummy** or **sf_homerun_dummy**.

In [83]:
# Regression modeling

laa_m1_stats = get_model_stats(laa_forward_df, 'forward', forward_targets, ['laa_homerun_dummy'])
lad_m1_stats = get_model_stats(lad_forward_df, 'forward', forward_targets, ['lad_homerun_dummy'])
cws_m1_stats = get_model_stats(cws_forward_df, 'forward', forward_targets, ['cws_homerun_dummy'])
sf_m1_stats = get_model_stats(sf_forward_df, 'forward', forward_targets, ['sf_homerun_dummy'])

laa_m1_stats_df = pd.DataFrame(laa_m1_stats)
lad_m1_stats_df = pd.DataFrame(lad_m1_stats)
cws_m1_stats_df = pd.DataFrame(cws_m1_stats)
sf_m1_stats_df = pd.DataFrame(sf_m1_stats)

In [84]:
# Plot home run beta for LAA

fig = graph_model_stats(
    df=laa_m1_stats_df,
    stat_type='beta',
    title='Model 1 Coefficient',
    z='laa_homerun_dummy',
    gradient='p_value',
    team='LAA'
)

fig.show()

In [85]:
# Plot home run beta for LAD

fig = graph_model_stats(
    df=lad_m1_stats_df,
    stat_type='beta',
    title='Model 1 Coefficient',
    z='lad_homerun_dummy',
    gradient='p_value',
    team='LAD'
)

fig.show()

In [86]:
# Plot home run beta for CWS

fig = graph_model_stats(
    df=cws_m1_stats_df,
    stat_type='beta',
    title='Model 1 Coefficient',
    z='cws_homerun_dummy',
    gradient='p_value',
    team='CWS'
)

fig.show()

In [87]:
# Plot home run beta for SF

fig = graph_model_stats(
    df=sf_m1_stats_df,
    stat_type='beta',
    title='Model 1 Coefficient',
    z='sf_homerun_dummy',
    gradient='p_value',
    team='SF'
)

fig.show()

In [88]:
# Plot R² for LAA

fig = graph_model_stats(
    df=laa_m1_stats_df,
    stat_type='r_squared',
    title='Model 1 R²',
    z='laa_homerun_dummy',
    gradient='r_squared',
    team='LAA'
)

fig.show()

In [89]:
# Plot R² for LAD

fig = graph_model_stats(
    df=lad_m1_stats_df,
    stat_type='r_squared',
    title='Model 1 R²',
    z='lad_homerun_dummy',
    gradient='r_squared',
    team='LAD'
)

fig.show()

In [90]:
# Plot R² for CWS

fig = graph_model_stats(
    df=cws_m1_stats_df,
    stat_type='r_squared',
    title='Model 1 R²',
    z='cws_homerun_dummy',
    gradient='r_squared',
    team='CWS'
)

fig.show()

In [91]:
# Plot R² for SF

fig = graph_model_stats(
    df=sf_m1_stats_df,
    stat_type='r_squared',
    title='Model 1 R²',
    z='sf_homerun_dummy',
    gradient='r_squared',
    team='SF'
)

fig.show()

### Model 2

$$\mathbb{E}[Y \mid X_1, X_2, X_3, X_4, X_5, X_6] = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_3 + \beta_4 X_4 + \beta_5 X_5 + \beta_6 X_6$$

where:

- $Y$ is **f_px_chg_w $\alpha$ _b $\beta$**,

- $X_1$ is either **laa_homerun_dummy**, **lad_homerun_dummy**, **cws_homerun_dummy** or **sf_homerun_dummy**,

- $X_2$ is the **price_at_hr**,

- $X_3$ is the **runners_on_base**,

- $X_4$ is the **inning**,

- $X_5$ is either **laa_score**, **lad_score**, **cws_score**, or **sf_score**, and

- $X_6$ is the **opp_score**.

In [92]:
# Regression modeling

laa_m2_stats = get_model_stats(laa_forward_df, 'forward', forward_targets, laa_forward_features)
lad_m2_stats = get_model_stats(lad_forward_df, 'forward', forward_targets, lad_forward_features)
cws_m2_stats = get_model_stats(cws_forward_df, 'forward', forward_targets, cws_forward_features)
sf_m2_stats = get_model_stats(sf_forward_df, 'forward', forward_targets, sf_forward_features)

laa_m2_stats_df = pd.DataFrame(laa_m2_stats)
lad_m2_stats_df = pd.DataFrame(lad_m2_stats)
cws_m2_stats_df = pd.DataFrame(cws_m2_stats)
sf_m2_stats_df = pd.DataFrame(sf_m2_stats)

In [93]:
# Plot home run beta for LAA

fig = graph_model_stats(
    df=laa_m2_stats_df,
    stat_type='beta',
    title='Model 2 Coefficient',
    z='laa_homerun_dummy',
    gradient='p_value',
    team='LAA'
)

fig.show()

In [94]:
# Plot home run beta for LAD

fig = graph_model_stats(
    df=lad_m2_stats_df,
    stat_type='beta',
    title='Model 2 Coefficient',
    z='lad_homerun_dummy',
    gradient='p_value',
    team='LAD'
)

fig.show()

In [95]:
# Plot home run beta for CWS

fig = graph_model_stats(
    df=cws_m2_stats_df,
    stat_type='beta',
    title='Model 2 Coefficient',
    z='cws_homerun_dummy',
    gradient='p_value',
    team='CWS'
)

fig.show()

In [96]:
# Plot home run beta for SF

fig = graph_model_stats(
    df=sf_m2_stats_df,
    stat_type='beta',
    title='Model 2 Coefficient',
    z='sf_homerun_dummy',
    gradient='p_value',
    team='SF'
)

fig.show()

In [ ]:
# Plot R² for LAA

fig = graph_model_stats(
    df=laa_m2_stats_df,
    stat_type='r_squared',
    title='Model 2 R²',
    z='laa_homerun_dummy',
    gradient='r_squared',
    team='LAA'
)

fig.show()

In [ ]:
# Plot R² for LAD

fig = graph_model_stats(
    df=lad_m2_stats_df,
    stat_type='r_squared',
    title='Model 2 R²',
    z='lad_homerun_dummy',
    gradient='r_squared',
    team='LAD'
)

fig.show()

In [ ]:
# Plot R² for CWS

fig = graph_model_stats(
    df=cws_m2_stats_df,
    stat_type='r_squared',
    title='Model 2 R²',
    z='cws_homerun_dummy',
    gradient='r_squared',
    team='CWS'
)

fig.show()

In [ ]:
# Plot R² for SF

fig = graph_model_stats(
    df=sf_m2_stats_df,
    stat_type='r_squared',
    title='Model 2 R²',
    z='sf_homerun_dummy',
    gradient='r_squared',
    team='SF'
)

fig.show()

## Double Target Regression Analysis

In [101]:
# Define features and targets

double_px_features = [col for col in laa_double_df.columns if 'd_price' in col]

laa_double_features = [
    'laa_homerun_dummy', 'runners_on_base', 
    'inning', 'laa_score', 'opp_score'
] + double_px_features
lad_double_features = [
    'lad_homerun_dummy', 'runners_on_base', 
    'inning', 'lad_score', 'opp_score'
] + double_px_features
cws_double_features = [
    'cws_homerun_dummy', 'runners_on_base', 
    'inning', 'cws_score', 'opp_score'
] + double_px_features
sf_double_features = [
    'sf_homerun_dummy', 'runners_on_base', 
    'inning', 'sf_score', 'opp_score'
] + double_px_features

double_targets = [col for col in laa_double_df.columns if 'd_px_chg' in col]

# Look at a subset of the features, and drop any price change data if missing
laa_double_df = laa_double_df[double_targets + laa_double_features].dropna(subset=double_targets)
lad_double_df = lad_double_df[double_targets + lad_double_features].dropna(subset=double_targets)
cws_double_df = cws_double_df[double_targets + cws_double_features].dropna(subset=double_targets)
sf_double_df = sf_double_df[double_targets + sf_double_features].dropna(subset=double_targets)

laa_double_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 298 entries, 0 to 362
Data columns (total 41 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   d_px_chg_w5_b2     298 non-null    float64
 1   d_px_chg_w4_b4     298 non-null    float64
 2   d_px_chg_w2_b3     298 non-null    float64
 3   d_px_chg_w5_b6     298 non-null    float64
 4   d_px_chg_w2_b5     298 non-null    float64
 5   d_px_chg_w1_b5     298 non-null    float64
 6   d_px_chg_w5_b1     298 non-null    float64
 7   d_px_chg_w2_b1     298 non-null    float64
 8   d_px_chg_w2_b2     298 non-null    float64
 9   d_px_chg_w3_b1     298 non-null    float64
 10  d_px_chg_w6_b1     298 non-null    float64
 11  d_px_chg_w6_b4     298 non-null    float64
 12  d_px_chg_w4_b2     298 non-null    float64
 13  d_px_chg_w5_b4     298 non-null    float64
 14  d_px_chg_w3_b5     298 non-null    float64
 15  d_px_chg_w3_b4     298 non-null    float64
 16  d_px_chg_w6_b3     298 non-null

In [102]:
lad_double_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 350 entries, 0 to 402
Data columns (total 41 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   d_px_chg_w5_b2     350 non-null    float64
 1   d_px_chg_w4_b4     350 non-null    float64
 2   d_px_chg_w2_b3     350 non-null    float64
 3   d_px_chg_w5_b6     350 non-null    float64
 4   d_px_chg_w2_b5     350 non-null    float64
 5   d_px_chg_w1_b5     350 non-null    float64
 6   d_px_chg_w5_b1     350 non-null    float64
 7   d_px_chg_w2_b1     350 non-null    float64
 8   d_px_chg_w2_b2     350 non-null    float64
 9   d_px_chg_w3_b1     350 non-null    float64
 10  d_px_chg_w6_b1     350 non-null    float64
 11  d_px_chg_w6_b4     350 non-null    float64
 12  d_px_chg_w4_b2     350 non-null    float64
 13  d_px_chg_w5_b4     350 non-null    float64
 14  d_px_chg_w3_b5     350 non-null    float64
 15  d_px_chg_w3_b4     350 non-null    float64
 16  d_px_chg_w6_b3     350 non-null

In [103]:
cws_double_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 226 entries, 4 to 285
Data columns (total 41 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   d_px_chg_w5_b2     226 non-null    float64
 1   d_px_chg_w4_b4     226 non-null    float64
 2   d_px_chg_w2_b3     226 non-null    float64
 3   d_px_chg_w5_b6     226 non-null    float64
 4   d_px_chg_w2_b5     226 non-null    float64
 5   d_px_chg_w1_b5     226 non-null    float64
 6   d_px_chg_w5_b1     226 non-null    float64
 7   d_px_chg_w2_b1     226 non-null    float64
 8   d_px_chg_w2_b2     226 non-null    float64
 9   d_px_chg_w3_b1     226 non-null    float64
 10  d_px_chg_w6_b1     226 non-null    float64
 11  d_px_chg_w6_b4     226 non-null    float64
 12  d_px_chg_w4_b2     226 non-null    float64
 13  d_px_chg_w5_b4     226 non-null    float64
 14  d_px_chg_w3_b5     226 non-null    float64
 15  d_px_chg_w3_b4     226 non-null    float64
 16  d_px_chg_w6_b3     226 non-null

In [104]:
sf_double_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 226 entries, 0 to 271
Data columns (total 41 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   d_px_chg_w5_b2    226 non-null    float64
 1   d_px_chg_w4_b4    226 non-null    float64
 2   d_px_chg_w2_b3    226 non-null    float64
 3   d_px_chg_w5_b6    226 non-null    float64
 4   d_px_chg_w2_b5    226 non-null    float64
 5   d_px_chg_w1_b5    226 non-null    float64
 6   d_px_chg_w5_b1    226 non-null    float64
 7   d_px_chg_w2_b1    226 non-null    float64
 8   d_px_chg_w2_b2    226 non-null    float64
 9   d_px_chg_w3_b1    226 non-null    float64
 10  d_px_chg_w6_b1    226 non-null    float64
 11  d_px_chg_w6_b4    226 non-null    float64
 12  d_px_chg_w4_b2    226 non-null    float64
 13  d_px_chg_w5_b4    226 non-null    float64
 14  d_px_chg_w3_b5    226 non-null    float64
 15  d_px_chg_w3_b4    226 non-null    float64
 16  d_px_chg_w6_b3    226 non-null    float64
 17  d_

### Model 1

$$\mathbb{E}[Y \mid X_1] = \beta_0 + \beta_1 X_1$$

where:

- $Y$ is **d_px_chg_w $\alpha$ _b $\beta$**, and

- $X_1$ is either **laa_homerun_dummy**, **lad_homerun_dummy**, **cws_homerun_dummy** or **sf_homerun_dummy**.

In [105]:
# Regression modeling

laa_m1_stats = get_model_stats(laa_double_df, 'double', double_targets, ['laa_homerun_dummy'])
lad_m1_stats = get_model_stats(lad_double_df, 'double', double_targets, ['lad_homerun_dummy'])
cws_m1_stats = get_model_stats(cws_double_df, 'double', double_targets, ['cws_homerun_dummy'])
sf_m1_stats = get_model_stats(sf_double_df, 'double', double_targets, ['sf_homerun_dummy'])

laa_m1_stats_df = pd.DataFrame(laa_m1_stats)
lad_m1_stats_df = pd.DataFrame(lad_m1_stats)
cws_m1_stats_df = pd.DataFrame(cws_m1_stats)
sf_m1_stats_df = pd.DataFrame(sf_m1_stats)

In [106]:
# Plot home run beta for LAA

fig = graph_model_stats(
    df=laa_m1_stats_df,
    stat_type='beta',
    title='Model 1 Coefficient',
    z='laa_homerun_dummy',
    gradient='p_value',
    team='LAA'
)

fig.show()

In [107]:
# Plot home run beta for LAD

fig = graph_model_stats(
    df=lad_m1_stats_df,
    stat_type='beta',
    title='Model 1 Coefficient',
    z='lad_homerun_dummy',
    gradient='p_value',
    team='LAD'
)

fig.show()

In [108]:
# Plot home run beta for CWS

fig = graph_model_stats(
    df=cws_m1_stats_df,
    stat_type='beta',
    title='Model 1 Coefficient',
    z='cws_homerun_dummy',
    gradient='p_value',
    team='CWS'
)

fig.show()

In [109]:
# Plot home run beta for SF

fig = graph_model_stats(
    df=sf_m1_stats_df,
    stat_type='beta',
    title='Model 1 Coefficient',
    z='sf_homerun_dummy',
    gradient='p_value',
    team='SF'
)

fig.show()

In [ ]:
# Plot R² for LAA

fig = graph_model_stats(
    df=laa_m1_stats_df,
    stat_type='r_squared',
    title='Model 1 R²',
    z='laa_homerun_dummy',
    gradient='r_squared',
    team='LAA'
)

fig.show()

In [ ]:
# Plot R² for LAD

fig = graph_model_stats(
    df=lad_m1_stats_df,
    stat_type='r_squared',
    title='Model 1 R²',
    z='lad_homerun_dummy',
    gradient='r_squared',
    team='LAD'
)

fig.show()

In [ ]:
# Plot R² for CWS

fig = graph_model_stats(
    df=cws_m1_stats_df,
    stat_type='r_squared',
    title='Model 1 R²',
    z='cws_homerun_dummy',
    gradient='r_squared',
    team='CWS'
)

fig.show()

In [ ]:
# Plot R² for SF

fig = graph_model_stats(
    df=sf_m1_stats_df,
    stat_type='r_squared',
    title='Model 1 R²',
    z='sf_homerun_dummy',
    gradient='r_squared',
    team='SF'
)

fig.show()

### Model 2

$$\mathbb{E}[Y \mid X_1, X_2, X_3, X_4, X_5, X_6] = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_3 + \beta_4 X_4 + \beta_5 X_5 + \beta_6 X_6$$

where:

- $Y$ is **d_px_chg_w $\alpha$ _b $\beta$**,

- $X_1$ is either **laa_homerun_dummy**, **lad_homerun_dummy**, **cws_homerun_dummy** or **sf_homerun_dummy**,

- $X_2$ is the **runners_on_base**,

- $X_3$ is the **inning**,

- $X_4$ is either **laa_score**, **lad_score**, **cws_score**, or **sf_score**,

- $X_5$ is the **opp_score**, and

- $X_6$ is the **d_price_w $\alpha$ _b $\beta$**.

In [114]:
# Regression modeling

laa_m2_stats = get_model_stats(laa_double_df, 'double', double_targets, laa_double_features)
lad_m2_stats = get_model_stats(lad_double_df, 'double', double_targets, lad_double_features)
cws_m2_stats = get_model_stats(cws_double_df, 'double', double_targets, cws_double_features)
sf_m2_stats = get_model_stats(sf_double_df, 'double', double_targets, sf_double_features)

laa_m2_stats_df = pd.DataFrame(laa_m2_stats)
lad_m2_stats_df = pd.DataFrame(lad_m2_stats)
cws_m2_stats_df = pd.DataFrame(cws_m2_stats)
sf_m2_stats_df = pd.DataFrame(sf_m2_stats)

In [115]:
# Plot home run beta for LAA

fig = graph_model_stats(
    df=laa_m2_stats_df,
    stat_type='beta',
    title='Model 2 Coefficient',
    z='laa_homerun_dummy',
    gradient='p_value',
    team='LAA'
)

fig.show()

In [116]:
# Plot home run beta for LAD

fig = graph_model_stats(
    df=lad_m2_stats_df,
    stat_type='beta',
    title='Model 2 Coefficient',
    z='lad_homerun_dummy',
    gradient='p_value',
    team='LAD'
)

fig.show()

In [117]:
# Plot home run beta for CWS

fig = graph_model_stats(
    df=cws_m2_stats_df,
    stat_type='beta',
    title='Model 2 Coefficient',
    z='cws_homerun_dummy',
    gradient='p_value',
    team='CWS'
)

fig.show()

In [118]:
# Plot home run beta for SF

fig = graph_model_stats(
    df=sf_m2_stats_df,
    stat_type='beta',
    title='Model 2 Coefficient',
    z='sf_homerun_dummy',
    gradient='p_value',
    team='SF'
)

fig.show()

In [119]:
# Plot R² for LAA

fig = graph_model_stats(
    df=laa_m2_stats_df,
    stat_type='r_squared',
    title='Model 2 R²',
    z='laa_homerun_dummy',
    gradient='r_squared',
    team='LAA'
)

fig.show()

In [120]:
# Plot R² for LAD

fig = graph_model_stats(
    df=lad_m2_stats_df,
    stat_type='r_squared',
    title='Model 2 R²',
    z='lad_homerun_dummy',
    gradient='r_squared',
    team='LAD'
)

fig.show()

In [121]:
# Plot R² for CWS

fig = graph_model_stats(
    df=cws_m2_stats_df,
    stat_type='r_squared',
    title='Model 2 R²',
    z='cws_homerun_dummy',
    gradient='r_squared',
    team='CWS'
)

fig.show()

In [122]:
# Plot R² for SF

fig = graph_model_stats(
    df=sf_m2_stats_df,
    stat_type='r_squared',
    title='Model 2 R²',
    z='sf_homerun_dummy',
    gradient='r_squared',
    team='SF'
)

fig.show()

## Summary of Hypothesis Testing

### Generic Target
Models using the Generic method performed the worst; however, the **laa_homerun_dummy**, **lad_homerun_dummy**, **cws_homerun_dummy**, and **sf_homerun_dummy** coefficients were significant across both model one and model two.
- **Model 1** has a decent $R^2$ value for LAA, CWS, and SF of 0.31, 0.33, and 0.28, respectively, and weak performance for LAD at 0.15.
- **Model 2** shows improved $R^2$ values for all teams: 0.40 for LAA, 0.26 for LAD, 0.44 for CWS, and 0.37 for SF.

### Forward Target
Models using the Forward Binned Mean method performed moderately, with the **laa_homerun_dummy**, **lad_homerun_dummy**, **cws_homerun_dummy**, and **sf_homerun_dummy** coefficients remaining significant across both model one and model two.
- **Model 1** yields moderate $R^2$ values across thirty-six window-bin combinations: LAA ranges from 0.31 to 0.47, LAD ranges from 0.15 to 0.32, CWS ranges from 0.34 to 0.38, and SF ranges from 0.31 to 0.44.
- **Model 2** yields improved $R^2$ values across thirty-six window-bin combinations: LAA ranges from 0.41 to 0.53, LAD ranges from 0.28 to 0.43, CWS ranges from 0.43 to 0.48, and SF ranges from 0.39 to 0.52.

### Double Target
Models using the Double Binned Mean method performed the best, with the **laa_homerun_dummy**, **lad_homerun_dummy**, **cws_homerun_dummy**, and **sf_homerun_dummy** coefficients remaining significant across both model one and model two.
- **Model 1** shows strong $R^2$ values across thirty-six window-bin combinations: LAA ranges from 0.31 to 0.67, LAD ranges from 0.14 to 0.56, CWS ranges from 0.34 to 0.56, and SF ranges from 0.29 to 0.67.
- **Model 2** shows slightly stronger $R^2$ values across thirty-six window-bin combinations: LAA ranges from 0.31 to 0.68, LAD ranges from 0.16 to 0.59, CWS ranges from 0.35 to 0.58, and SF ranges from 0.30 to 0.68.